# Capstone, actions that survive

**Scenario:** a triage assistant books urgent imaging slots. A radiologist notices the same patient
in two CT slots an hour apart. Nothing in the logs looks wrong. The agent asked once. The runtime
booked twice.

The cause is a retry, and the fix is **a numbered ticket at a deli counter**. You take a ticket once.
Waving it again does not make a second sandwich.

This capstone closes the vault. An action has to survive two things: a retry within a run, and a
restart between runs. Both come down to the same question, which is whether anything remembers.

## Mechanics

After you run a tool, you tell the model what happened by appending a message. Its shape matters.

| Field | Value | Why |
|---|---|---|
| `role` | `"tool"` | On OpenAI shaped APIs. Anthropic puts the result in a `user` message instead |
| `tool_call_id` | the `id` from the request | How the model matches result to request. Wrong id, wrong reasoning |
| `content` | a string | Not an object. Serialise it yourself |

Two things follow. The model **cannot see your side effects**, only the string you hand back. And
`tool_call_id` already names one intended action, which is what an idempotency key needs to be.

## The picture

![The same call id must never book twice](images/idempotency.svg)

The ledger sits between the decision and the booking, and it is the only thing that knows an action
already happened.

## The cost

```
harm = duplicate bookings x (wasted slot + patient recalled + radiologist hour)
```

Unlike a token bill, this one is paid by somebody in a waiting room.

## The failure

A booking backend, and a runtime that retries when the network wobbles.

In [1]:
import json
from vault import get_client, load_env, model_for

load_env()
client = get_client("01-stateful-agent-runtime/03-capstone-actions-that-survive")

BOOKED = []          # stands in for the scheduling system
FAIL_ONCE = {"left": 1}


def book_scan(patient_id, modality, urgency):
    """Book a slot. Fails the first time, the way a flaky network does."""
    if FAIL_ONCE["left"] > 0:
        FAIL_ONCE["left"] -= 1
        raise TimeoutError("scheduler did not respond")
    BOOKED.append({"patient_id": patient_id, "modality": modality, "urgency": urgency})
    return {"slot": f"SLOT-{len(BOOKED):03d}"}

The timeout is the honest part. The call may have reached the scheduler and the reply may have been
lost, so the runtime does not know whether the booking happened.

In [2]:
BOOK_TOOL = {"type": "function", "function": {
    "name": "book_scan",
    "description": "Book an imaging slot for a patient.",
    "parameters": {"type": "object", "properties": {
        "patient_id": {"type": "string"},
        "modality": {"type": "string", "enum": ["ct", "mri", "xray"]},
        "urgency": {"type": "string", "enum": ["routine", "urgent"]}},
        "required": ["patient_id", "modality", "urgency"],
        "additionalProperties": False}}}

reply = client.chat.completions.create(
    model=model_for("default"), max_tokens=300, tools=[BOOK_TOOL],
    messages=[{"role": "system", "content": "You triage radiology referrals."},
              {"role": "user", "content": "Patient P-4471, suspected stroke, needs a CT now."}])

call = reply.choices[0].message.tool_calls[0]
args = json.loads(call.function.arguments)
print(f"model asked for: {call.function.name}({args})")
print(f"call id        : {call.id}")

model asked for: book_scan({'urgency': 'urgent', 'modality': 'ct', 'patient_id': 'P-4471'})
call id        : tool_book_scan_o3Qm7OjLgHlyekxWfSsw


Now the retry loop most runtimes ship with. Careful about failure, careless about repetition.

In [3]:
def run_with_retry(name, args, attempts=3):
    """Retry a flaky tool. Nothing here remembers a previous attempt."""
    for attempt in range(attempts):
        try:
            return book_scan(**args)
        except TimeoutError:
            print(f"  attempt {attempt + 1} timed out, retrying")
    raise RuntimeError("gave up")


result = run_with_retry(call.function.name, args)
print(f"\nbooked: {result}")
print(f"rows in the scheduler: {len(BOOKED)}")

  attempt 1 timed out, retrying

booked: {'slot': 'SLOT-001'}
rows in the scheduler: 1


One booking, because the first attempt raised before it wrote. Now the harder version, where the
write lands and the reply is lost.

In [4]:
BOOKED.clear()


def book_scan_lossy(patient_id, modality, urgency, drop_reply=True):
    """Writes, then loses the response. The caller cannot tell."""
    BOOKED.append({"patient_id": patient_id, "modality": modality, "urgency": urgency})
    if drop_reply and len(BOOKED) == 1:
        raise TimeoutError("scheduler did not respond")
    return {"slot": f"SLOT-{len(BOOKED):03d}"}


for attempt in range(2):
    try:
        book_scan_lossy(**args)
        break
    except TimeoutError:
        print(f"  attempt {attempt + 1} timed out, retrying")

print(f"\nrows in the scheduler: {len(BOOKED)}")
assert len(BOOKED) == 1, f"patient booked {len(BOOKED)} times for one request"

  attempt 1 timed out, retrying

rows in the scheduler: 2


AssertionError: patient booked 2 times for one request

## The diagnosis

The patient is booked twice, and every part behaved correctly.

The model asked once. The retry was right, because a timeout does not tell you whether the write
landed. The backend did what it was told, twice.

What is missing is memory. **Nothing recorded that this action was already attempted**, so the second
attempt looks like a first. Retries are not optional, so the duplicate is not a bug in the retry. It
is a missing property in the thing being retried.

The identifier was already in hand. `tool_call_id` names one action and does not change on retry.

## The fix

The tempting fix is a ledger in the runtime, written after the call returns. It does not work. If the
write lands and the reply is lost, the ledger never gets the entry, and the retry duplicates exactly
as before.

**The key has to travel with the request**, so the side that does the writing is the side that
decides whether it already wrote.

In [5]:
SCHEDULER_KEYS = {}      # lives with the scheduler, not with the agent


def book_scan_idempotent(patient_id, modality, urgency, idempotency_key,
                         drop_reply=False):
    """Book at most once per key. The dedupe happens before the write."""
    if idempotency_key in SCHEDULER_KEYS:
        return SCHEDULER_KEYS[idempotency_key]
    BOOKED.append({"patient_id": patient_id, "modality": modality, "urgency": urgency})
    outcome = {"slot": f"SLOT-{len(BOOKED):03d}"}
    SCHEDULER_KEYS[idempotency_key] = outcome
    if drop_reply:
        raise TimeoutError("scheduler did not respond")
    return outcome

The write and the key land together, before the reply can be lost. Same broken network.

In [6]:
BOOKED.clear()
SCHEDULER_KEYS.clear()

for attempt in range(2):
    try:
        # Reply is dropped on the first attempt, after the row is written.
        outcome = book_scan_idempotent(**args, idempotency_key=call.id,
                                       drop_reply=(attempt == 0))
        print(f"  attempt {attempt + 1} returned {outcome}")
        break
    except TimeoutError:
        print(f"  attempt {attempt + 1} timed out, retrying with the same key")

print(f"\nrows in the scheduler: {len(BOOKED)}")
print(f"before the fix: 2 rows for one request")
print(f"after the fix : {len(BOOKED)} row for one request")

  attempt 1 timed out, retrying with the same key
  attempt 2 returned {'slot': 'SLOT-001'}

rows in the scheduler: 1
before the fix: 2 rows for one request
after the fix : 1 row for one request


Two pieces remain in the production shape. The key must be stable, and the model must be told what
happened.

In [7]:
import hashlib


def idempotency_key(call_id, args):
    """Stable for one intended action, and unchanged across retries."""
    payload = json.dumps(args, sort_keys=True)
    return f"{call_id}:{hashlib.sha256(payload.encode()).hexdigest()[:12]}"

Hashing the arguments alongside the id means a retry reuses the key while a different action never
collides with it. Then the result message, which is all the model ever sees.

In [8]:
def tool_result_message(call_id, outcome):
    """What the model gets back. The id is how it matches this to its request."""
    return {"role": "tool",
            "tool_call_id": call_id,
            "content": json.dumps(outcome)}


key = idempotency_key(call.id, args)
message = tool_result_message(call.id, SCHEDULER_KEYS[call.id])
print(f"key    : {key}")
print(f"message: {message}")

key    : tool_book_scan_o3Qm7OjLgHlyekxWfSsw:9a506228a02c
message: {'role': 'tool', 'tool_call_id': 'tool_book_scan_o3Qm7OjLgHlyekxWfSsw', 'content': '{"slot": "SLOT-001"}'}


That handles a retry inside one run. Now the second half, which is a restart between runs.

In [9]:
import pathlib

STATE_FILE = pathlib.Path("runtime-state.json")


def load_ledger():
    """The ledger a restart can still read."""
    if STATE_FILE.is_file():
        return json.loads(STATE_FILE.read_text())
    return {}

The in-memory dict from a moment ago dies with the process. A key that does not outlive the runtime
is a cache, not an idempotency key. Writing it to disk is the smallest thing that fixes that, and the
shape is identical for Redis or Postgres.

In [10]:
def book_durable(patient_id, modality, urgency, key):
    """Book at most once per key, across restarts as well as retries."""
    ledger = load_ledger()
    if key in ledger:
        return ledger[key], "replayed"
    BOOKED.append({"patient_id": patient_id, "modality": modality, "urgency": urgency})
    ledger[key] = {"slot": f"SLOT-{len(BOOKED):03d}"}
    STATE_FILE.write_text(json.dumps(ledger, indent=2))
    return ledger[key], "executed"

Book, then simulate the process dying and coming back, which here is simply calling it again with
nothing held in memory.

In [11]:
BOOKED.clear()
STATE_FILE.unlink(missing_ok=True)

first, how_first = book_durable(**args, key=key)
second, how_second = book_durable(**args, key=key)      # after a restart

print(f"first call : {first} ({how_first})")
print(f"after restart: {second} ({how_second})")
print(f"rows in the scheduler: {len(BOOKED)}")

first call : {'slot': 'SLOT-001'} (executed)
after restart: {'slot': 'SLOT-001'} (replayed)
rows in the scheduler: 1


## The gate

One test for both properties, because they are the same property twice: an action happens once,
however many times the runtime asks.

In [12]:
def test_one_action_happens_once():
    BOOKED.clear()
    STATE_FILE.unlink(missing_ok=True)
    for _ in range(5):                       # retries and restarts alike
        book_durable("P-1", "ct", "urgent", key="same-key")
    assert len(BOOKED) == 1, f"booked {len(BOOKED)} times for one key"


test_one_action_happens_once()
STATE_FILE.unlink(missing_ok=True)
print("gate holds: five attempts across restarts, one booking")

gate holds: five attempts across restarts, one booking


Point `load_ledger` at a fresh dict instead of the file and this fails on the second attempt.

### Enterprise exploration

- A file works for one process. Where does the ledger live across four replicas and a deploy?
- The row is written before the ledger entry. What if the process dies between them, and how do you
  shrink that window?
- Keys cannot be kept forever. What is your retention, and what does a retry look like after expiry?
- Duplicate scans waste scanner time and recall patients, and duplicate patient records are
  reportable in most health jurisdictions. What is the compliance exposure of finding this a month
  late, and how would you detect it in production instead?

### Key takeaways

- A timeout does not tell you whether the write landed. Retries must be safe by construction.
- A ledger written after success does not survive a lost reply. The key travels with the request.
- `tool_call_id` names one intended action and is stable across retries.
- State that dies with the process makes the key decorative.